# 14.2 Data Drift

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 14: MLOps & Monitoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-14-mlops-and-monitoring/14.02_data-drift.ipynb)

Companion code for Section 14.2 of the book. Run the cells in order from top to bottom.

**Example 14.2.1** · bash

```bash
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import ks_2samp, chi2_contingency
from typing import Dict, Tuple, Optional
import warnings; warnings.filterwarnings('ignore')

# ── Complete Drift Detection Framework ───────────────────────────
class DataDriftDetector:
    '''
    Detects statistical drift in feature distributions.
    Supports PSI (Population Stability Index), KS test, Chi-squared.
    '''

    def __init__(self,
                  psi_warning_threshold: float = 0.1,
                  psi_alert_threshold:   float = 0.25,
                  ks_significance:       float = 0.05,
                  chi2_significance:     float = 0.05,
                  n_bins:                int   = 10):
        self.psi_warning  = psi_warning_threshold
        self.psi_alert    = psi_alert_threshold
        self.ks_sig       = ks_significance
        self.chi2_sig     = chi2_significance
        self.n_bins       = n_bins

    # ── PSI ────────────────────────────────────────────────────────
    def compute_psi(self,
                     reference: np.ndarray,
                     current:   np.ndarray,
                     n_bins: Optional[int] = None) -> Dict:
        n_bins = n_bins or self.n_bins
        eps = 1e-8   # Prevent log(0)

        # Compute bins on reference data
        bins = np.percentile(reference, np.linspace(0, 100, n_bins+1))
        bins[0]  -= 1e-6
        bins[-1] += 1e-6

        # Proportion in each bin
        ref_counts, _ = np.histogram(reference, bins=bins)
        cur_counts, _ = np.histogram(current,   bins=bins)

        ref_pct = ref_counts / len(reference) + eps
        cur_pct = cur_counts / len(current)   + eps

        # PSI formula
        psi_per_bin = (cur_pct - ref_pct) * np.log(cur_pct / ref_pct)
        psi_total   = psi_per_bin.sum()

        return {
            'psi':         float(psi_total),
            'psi_per_bin': psi_per_bin.tolist(),
            'bins':        bins.tolist(),
            'status':      self._psi_status(psi_total),
        }

    def _psi_status(self, psi: float) -> str:
        if psi < self.psi_warning: return 'OK'
        if psi < self.psi_alert:   return 'WARNING'
        return 'ALERT'

    # ── KS Test ────────────────────────────────────────────────────
    def ks_test(self, reference: np.ndarray, current: np.ndarray) -> Dict:
        stat, p_value = ks_2samp(reference, current)
        drifted = p_value < self.ks_sig
        return {
            'ks_statistic': float(stat),
            'p_value':      float(p_value),
            'drifted':      drifted,
            'status':       'ALERT' if drifted else 'OK',
        }

    # ── Chi-Squared (categorical) ──────────────────────────────────
    def chi2_test(self, reference: pd.Series, current: pd.Series) -> Dict:
        all_cats = set(reference.unique()) | set(current.unique())
        ref_counts = pd.Series({c: (reference==c).sum() for c in all_cats})
        cur_counts = pd.Series({c: (current==c).sum()   for c in all_cats})

        contingency = pd.DataFrame({'reference': ref_counts, 'current': cur_counts}).fillna(0)
        chi2, p, dof, _ = chi2_contingency(contingency)
        drifted = p < self.chi2_sig
        return {
            'chi2_statistic': float(chi2),
            'p_value':        float(p),
            'dof':            int(dof),
            'drifted':        drifted,
            'status':         'ALERT' if drifted else 'OK',
        }

    # ── Full report ────────────────────────────────────────────────
    def generate_report(self,
                          reference_df: pd.DataFrame,
                          current_df:   pd.DataFrame,
                          numeric_cols: list,
                          categorical_cols: list = None) -> pd.DataFrame:
        rows = []
        categorical_cols = categorical_cols or []

        for col in numeric_cols:
            ref = reference_df[col].dropna().values
            cur = current_df[col].dropna().values

            psi_res = self.compute_psi(ref, cur)
            ks_res  = self.ks_test(ref, cur)

            overall_status = 'ALERT' if (psi_res['status']=='ALERT' or ks_res['drifted']) else \
                             'WARNING' if psi_res['status']=='WARNING' else 'OK'

            rows.append({
                'feature':    col,
                'type':       'numeric',
                'psi':        round(psi_res['psi'], 4),
                'psi_status': psi_res['status'],
                'ks_stat':    round(ks_res['ks_statistic'], 4),
                'ks_p':       round(ks_res['p_value'], 4),
                'ks_drifted': ks_res['drifted'],
                'status':     overall_status,
                'ref_mean':   round(ref.mean(), 4),
                'cur_mean':   round(cur.mean(), 4),
                'mean_shift': round((cur.mean()-ref.mean())/ref.std(), 4),
            })

        for col in categorical_cols:
            chi2_res = self.chi2_test(reference_df[col], current_df[col])
            rows.append({
                'feature':    col,
                'type':       'categorical',
                'chi2_stat':  round(chi2_res['chi2_statistic'], 4),
                'chi2_p':     round(chi2_res['p_value'], 4),
                'status':     chi2_res['status'],
            })

        return pd.DataFrame(rows).set_index('feature')


# ── Simulate drift scenario ──────────────────────────────────────
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
df   = pd.DataFrame(data.data, columns=data.feature_names)

np.random.seed(42)
n = len(df)

# Reference: training distribution
reference_df = df.copy()

# Current: drifted production data
current_df = df.copy()
# Simulate drift: shift 5 features, scale 3 others
drift_features = list(data.feature_names[:5])
stable_features = list(data.feature_names[5:])

for i, col in enumerate(drift_features):
    shift = reference_df[col].std() * (1.0 + 0.3*i)  # Increasing shift
    current_df[col] = current_df[col] + shift + np.random.normal(0, shift*0.3, n)

# Small random noise for stable features
for col in stable_features:
    current_df[col] += np.random.normal(0, reference_df[col].std()*0.05, n)

# Run drift detection
detector = DataDriftDetector()
report   = detector.generate_report(reference_df, current_df, list(data.feature_names))

print('=== Data Drift Report ===')
print(f'Features monitored: {len(report)}')
print(f'ALERT:   {(report.status=="ALERT").sum()}')
print(f'WARNING: {(report.status=="WARNING").sum()}')
print(f'OK:      {(report.status=="OK").sum()}')
print(f'\nDrift Report (numeric features):')
print(f'{'Feature':30s} {'PSI':>8} {'Status':>10} {'KS p-val':>10} {'Mean Shift':>12}')
print('='*74)
for feat, row in report.iterrows():
    if 'psi' in report.columns and pd.notna(row.get('psi')):
        status_icon = ' ' if row['status']=='ALERT' else ' ' if row['status']=='WARNING' else ' '
        print(f'{feat:30s} {row["psi"]:>8.4f} {status_icon}{row["status"]:>9} {row["ks_p"]:>10.4f}
{row["mean_shift"]:>12.4f}')

# Visualise drift
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Data Drift Detection - Feature Distributions', fontsize=14, fontweight='bold')

top_drifted = report[report['type']=='numeric'].nlargest(3, 'psi').index.tolist()
top_stable  = report[report['type']=='numeric'].nsmallest(3, 'psi').index.tolist()

for ax, feat, title_prefix in zip(axes[0], top_drifted, ['Drifted']*3):
    ax.hist(reference_df[feat], bins=30, alpha=0.6, color='#1976D2', density=True, label='Reference')
    ax.hist(current_df[feat],   bins=30, alpha=0.6, color='#E53935', density=True, label='Current')
    psi = report.loc[feat, 'psi']
    ax.set_title(f'{feat[:25]}\nPSI={psi:.4f}   DRIFTED', fontsize=9)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

for ax, feat, title_prefix in zip(axes[1], top_stable, ['Stable']*3):
    ax.hist(reference_df[feat], bins=30, alpha=0.6, color='#1976D2', density=True, label='Reference')
    ax.hist(current_df[feat],   bins=30, alpha=0.6, color='#43A047', density=True, label='Current')
    psi = report.loc[feat, 'psi']
    ax.set_title(f'{feat[:25]}\nPSI={psi:.4f}   STABLE', fontsize=9)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('data_drift_report.png', dpi=150, bbox_inches='tight')
plt.show()
```